In [ ]:
# SISO 5G gNB-UE Simulation using AIRSTRAN D 2200
import sys
import os

# Add src directory to Python path
sys.path.append(os.path.abspath('../src'))

# Import or install Sionna
try:
    import sionna.rt
except ImportError as e:
    os.system("pip install sionna-rt")
    import sionna.rt

# Other imports
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import mitsuba as mi
import warnings

# Suppress warnings
warnings.filterwarnings("ignore", message="invalid value encountered in multiply")
warnings.filterwarnings("ignore", category=UserWarning, module="jupyter_client")

# Import relevant components from Sionna RT
from sionna.rt import load_scene, Transmitter, Receiver, Camera, PathSolver
from sionna.rt import AntennaArray, PlanarArray, SceneObject, ITURadioMaterial
from sionna.rt.antenna_pattern import antenna_pattern_registry

scene_xml_path = "../scene/scenes/Duke/scene.xml"
scene = load_scene(scene_xml_path)

In [ ]:
# ============================================
# SISO Configuration: gNB to UE
# ============================================

scene.frequency = 3.65e9  # 3.7 GHz

# Define UE position (fixed to start)
ue_position = [10.0, 0.0, 0.0]   # UE position (x, y, z in meters)

# ============================================
# Antenna Configuration
# ============================================

# gNB antenna: 3GPP TR 38.901 pattern (AIRSTRAN D 2200)
gnb_pattern_factory = antenna_pattern_registry.get("tr38901")
gnb_pattern = gnb_pattern_factory(polarization="V")

# Creating the pattern for the friendly jammer
friendly_jammer = antenna_pattern_registry.get("iso")
friendly_pattern = friendly_jammer(polarization="V")

# Rx pattern used to take measurements
ue_pattern_factory = antenna_pattern_registry.get("iso")
ue_pattern = ue_pattern_factory(polarization="V")

# SISO: Single antenna element at origin [0, 0, 0] for both TX and RX
single_element = np.array([[0.0, 0.0, 0.0]])  # Shape: (1, 3)

# Configure antenna arrays
scene.tx_array = AntennaArray(
    antenna_pattern=gnb_pattern,
    normalized_positions=single_element.T  # Shape: (3, 1)
)

jammer_array = AntennaArray(
    antenna_pattern=friendly_pattern,
    normalized_positions=single_element.T
)

scene.rx_array = AntennaArray(
    antenna_pattern=ue_pattern,
    normalized_positions=single_element.T  # Shape: (3, 1)
)

# ============================================
# Add Receiver to Scene
# ============================================

# Create UE receiver
rx = Receiver(name="ue", position=ue_position, display_radius=0.03)
scene.add(rx)

# ============================================
# Configure Propagation Environment
# ============================================

# Disable scattering for basic simulation
for radio_material in scene.radio_materials.values():
    radio_material.scattering_coefficient = 0.4

In [ ]:
from scene_parser import extract_building_info
from tx_placement import TxPlacement
# ============================================
# Place gNB on a Specific Building
# ============================================

#building_info = extract_building_info(scene_xml_path, verbose=True)
# Old: 37
#selected_building_id = 33  # Change this to your desired building number
# TxPlacement will create the transmitter if it doesn't exist and place it on the building
# Correct parameter order: (scene, tx_name, scene_xml_path, building_id, offset)
#tx_placer = TxPlacement(scene, "gnb", scene_xml_path, selected_building_id, offset=30.0)
#tx_placer.set_rooftop_center()
# Get reference to the transmitter (already added to scene by TxPlacement)
#tx = tx_placer.tx
# Convert to flat numpy array instead of nested list
#gnb_position = tx.position.numpy().flatten().tolist()
# Point antenna toward UE
#tx.look_at(ue_position)
#print(f"\nSuccess! gNB placed on building {selected_building_id}")
#print(f"Position: {gnb_position}")

# Adding a transmitter (unconstrained)
gnb_position = [-40.0, 45.0, 30.0]
tx = Transmitter(name="tx", position=gnb_position)
scene.add(tx)

# ============================================
# Compute Propagation Paths
# ============================================

# Instantiate path solver
p_solver = PathSolver()

# Compute propagation paths
paths = p_solver(
    scene=scene,
    max_depth=5,
    los=True,
    specular_reflection=True,
    diffuse_reflection=False,
    refraction=False,
    seed=41
)

# ============================================
# Visualize Scene
# ============================================

# Setup camera
cam = Camera(position=(100.0, 100.0, 50.0))
cam.look_at(gnb_position)

# Preview the scene with propagation paths
#scene.preview(
#    paths=paths,
#    resolution=[1000, 1000],
#    clip_at=200,
#    show_orientations=True
#)

In [ ]:
from boresight_pathsolver import create_zone_mask
import numpy as np

map_config = {
    'center': [0.0, 0.0, 0.0],
    'size': [1400, 1400],
    'cell_size': (0.5, 0.5),
    'ground_height': 0.0,
}

extent = [
    map_config['center'][0] - map_config['size'][0] / 2,
    map_config['center'][0] + map_config['size'][0] / 2,
    map_config['center'][1] - map_config['size'][1] / 2,
    map_config['center'][1] + map_config['size'][1] / 2,
]

zone_params = {
    'center': [0.0, 0.0],
    'width': 250,
    'height': 250,
}

zone_mask, naive_look_at, zone_stats = create_zone_mask(
    map_config=map_config,
    zone_type='box',
    origin_point=gnb_position,
    zone_params=zone_params,
    target_height=0.0,
    scene_xml_path=scene_xml_path,
    exclude_buildings=True,
)
print(f"Zone contains {zone_stats['num_cells']} grid cells")
print(f"Zone coverage: {zone_stats['coverage_fraction']*100:.1f}% of map")
print(f"Naive baseline look-at: {zone_stats['look_at_xyz']}")
print(f"Zone centroid: {zone_stats['centroid_xy']}")

In [ ]:
from tx_placement import TxPlacement

#selected_building_id_2 = 21
#tx_placer2 = TxPlacement(scene, 'gnb2', scene_xml_path,
#                          selected_building_id_2, offset=30.0)
#tx_placer2.set_rooftop_center()
#tx2 = tx_placer2.tx
#tx2.look_at([0.0, 600.0, 0.0])

gnb2_position = [-40.0, -40.0, 30.0]
tx2 = Transmitter(name="tx2", position=gnb2_position)
scene.add(tx2)

gnb3_position = [100.0, -25.0, 30.0]
tx3 = Transmitter(name="tx3", position=gnb3_position)
scene.add(tx3)

#gnb2_position = tx2.position.numpy().flatten().tolist()
#print(f'gnb2 placed on building {selected_building_id_2}')
#print(f'Position: {gnb2_position}')

In [ ]:
from multi_tx_optimizer import TxConfig

# Build TxConfig list — order must match scene insertion order
tx_configs = [
    TxConfig(
        name='tx',
        on_building=False,
        building_id=1,
        zone_params=zone_params,
    ),
    TxConfig(
        name='tx2',
        on_building=False,
        building_id=2,
        zone_params=zone_params,
    ),
    TxConfig(
        name='tx3',
        on_building=False,
        building_id=3,
        zone_params=zone_params,
    )
]

In [ ]:
from multi_tx_optimizer import optimize_multi_tx, JammerConfig

from boresight_pathsolver import visualize_multi_tx_strata
import matplotlib.pyplot as plt

def strata_callback(iteration, tx_states, tx_configs, jam_positions=None):
    fig = visualize_multi_tx_strata(
        tx_states, tx_configs, map_config, iteration=iteration,
        jam_positions=jam_positions,
    )
    plt.show()
    plt.close(fig)

jam_configs = [
    #JammerConfig(name="jam_0", initial_power_dbm=23.0, initial_position=[0.0, 125.0]),
    JammerConfig(name="jam_1", initial_power_dbm=23.0, initial_position=[-225.0, 0.0]),
    JammerConfig(name="jam_2", initial_power_dbm=23.0, initial_position=[0.0, -225.0]),
    JammerConfig(name="jam_3", initial_power_dbm=23.0, initial_position=[225.0, 0.0]),
    #JammerConfig(name="jam_4", initial_power_dbm=23.0, initial_position=[0.0, 225.0])
]

multi_result, jam_scene = optimize_multi_tx(
    scene=scene,
    tx_configs=tx_configs,
    map_config=map_config,
    scene_xml_path=scene_xml_path,
    jammer_array=jammer_array,
    jam_configs=jam_configs,
    num_sample_points=500,
    learning_rate=3.0,
    num_iterations=50,
    noise_power=1e-10,
    dead_tail_percentile=60.0,
    max_dbscan_points=100_000,
    lds='Halton',
    sampler='rejection',
    debug_viz=False,
    sampling_strata='full',
    on_iteration_callback=strata_callback,
)

for tx_name, res in multi_result.items():
    if tx_name == 'joint':
        print(f'Final SIR loss: {res["loss_history"][-1]:.4f}')
        print(f'Elapsed: {res["elapsed_time_s"]:.1f}s')
        for jname, jd in res.get('jammers', {}).items():
            print(f'  {jname}: pos=({jd["final_position"][0]:.1f}, {jd["final_position"][1]:.1f}), '
                  f'pwr={jd["final_power_dbm"]:.1f} dBm')
    else:
        print(f'{tx_name}: Az={res["best_angles"][0]:.1f}°, '
              f'El={res["best_angles"][1]:.1f}°')
        print(f'  Position: ({res["final_position"][0]:.2f}, '
              f'{res["final_position"][1]:.2f})')

In [ ]:
from multi_tx_optimizer import compare_multi_tx_performance
import matplotlib.pyplot as plt
import json

zone_masks_dict = {
    'tx':  zone_mask,
    'tx2': zone_mask,
    'tx3': zone_mask
}

fig_cmp, comparison_stats = compare_multi_tx_performance(
    scene=scene,
    tx_configs=tx_configs,
    multi_result=multi_result,
    map_config=map_config,
    zone_masks=zone_masks_dict,
    noise_power=1e-10,
    jam_scene=jam_scene,
    jammer_configs=jam_configs,
)
plt.show()

# Print summary
def _pct(v): return f'{100*v:.1f}%' if v is not None else 'n/a'

for tx_name, s in comparison_stats.items():
    if tx_name == 'joint':
        continue
    c = s['containment']
    si = s['sinr_inside']
    so = s['sinr_outside']
    print(f'\n{tx_name}:')
    print(f'  rho_leak: {_pct(c["rho_leak"])}  (outside >= gamma)')
    print(f'  rho_hole: {_pct(c["rho_hole"])}  (inside  <  gamma)')
    if si: print(f'  SINR inside  mean/p10: {si["sinr_mean_db"]:+.1f} / {si["sinr_p10_db"]:+.1f} dB')
    if so: print(f'  SINR outside mean/p10: {so["sinr_mean_db"]:+.1f} / {so["sinr_p10_db"]:+.1f} dB')

# Save serialisable stats for offline analysis
save_stats = {k: v for k, v in comparison_stats.items()}
for tx_name, s in save_stats.items():
    if tx_name == 'joint':
        continue
    s.get('sinr_inside',  {}).pop('sinr_values_db', None)
    s.get('sinr_outside', {}).pop('sinr_values_db', None)

with open('multi_tx_comparison.json', 'w') as f:
    json.dump(save_stats, f, indent=2)
print('\nStats saved to multi_tx_comparison.json')